# 46 · 数据飞轮 + CI 集成 + Judge 校准

> **学习目标**：手撸数据飞轮（active learning / adversarial examples / hard cases）+ CI 集成（GitHub Actions / pre-commit）+ judge 校准（self-consistency / confidence calibration）。

> **预备**：24 评估 4 件套 + 44 RAGAS 5 指标 + 45 A/B 测试与可观测性已跑通。

> **为什么重要**：**数据飞轮** 是 RAG 工程师的核心竞争力：从失败 case 中学习，主动收集 hard cases，持续迭代 eval set。**CI 集成** 是自动化评估：每次 PR 自动跑 eval，防止退化。**Judge 校准** 是提升 LLM-as-judge 的稳定性：self-consistency（多采样）+ confidence calibration（置信度校准）。

> **生产 RAG 的「飞轮效应」**：
→ 每次改动 → eval 找到 weak cases → 人工标注 → 更新 eval set → 下次 eval 更准 → 验证改动更有效 → 飞轮加速迭代。

In [ ]:
MODE = 'OFFLINE'

import numpy as np, hashlib, re, json, time, requests
from dataclasses import dataclass, field
from typing import Callable
from math import isnan

OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text, dim=256):
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text):
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model':'nomic-embed-text','prompt':text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_chat(prompt, temp=0.0):
    r = requests.post(f'{OLLAMA}/api/chat', json={
        'model':'qwen1.5_1.8','stream':False,'options':{'temperature':temp},
        'messages':[{'role':'user','content':prompt}]
    }, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if MODE == 'ONLINE':
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); embed = ollama_embed; chat = ollama_chat; print('✅ ONLINE')
    except Exception: MODE='OFFLINE'
if MODE == 'OFFLINE':
    embed = fake_embed
    def chat(p): return f'[STUB-LLM] {p[:60]}'
    print('OFFLINE：metric 用规则评分；ONLINE 切到 LLM-as-judge')

## 1. 数据飞轮 —— 从失败 case 中学习

**数据飞轮的核心**：
- **主动收集**：从 live traffic 中收集 hard cases（low faithfulness / low answer_correctness）
- **人工标注**：人工标注 hard cases 的 label（正确答案 / 为什么错）
- **更新 eval set**：把 hard cases 加入 eval set，提高覆盖度
- **迭代验证**：下次 eval 看指标是否改善

**数据飞轮的 3 种类型**：

1. **Active Learning（主动学习）**
- 从 live traffic 中采样 low faithfulness 的 case
- 用 LLM-as-judge 评分 → 人工标注
- 加入 eval set，提高覆盖度

2. **Adversarial Examples（对抗样本）**
- 生成对抗 query（如同义改写 / 反问 / 歧义问题）
- 人工标注对抗样本的 label
- 加入 eval set，提高鲁棒性

3. **Hard Cases（困难样本）**
- 从 eval set 中找出 high faithfulness 但 low context_precision 的 case
- 分析为什么 recall 烂（chunking / embedding / index）
- 优化 retrieval 策略，提高召回

**数据飞轮的流程**：
```
1. Monitor（监控 live traffic）
   → 找出 low faithfulness / low answer_correctness 的 case
2. Sample（采样）
   → 抽取 20-50 条 hard cases
3. Label（标注）
   → 人工标注 label（正确答案 / 为什么错）
4. Add（加入 eval set）
   → 更新 eval set，提高覆盖度
5. Evaluate（迭代验证）
   → 下次 eval 看指标是否改善
6. Deploy（部署）
   → 验证改动是否有效
```

In [ ]:
# 数据飞轮数据结构

@dataclass
class HardCase:
    trace_id: str
    question: str
    answer: str
    contexts: list[str]
    faithfulness: float
    answer_correctness: float
    label: str | None = None  # 人工标注：正确答案 / 为什么错
    timestamp: float = 0.0

@dataclass
class EvalSet:
    name: str
    cases: list[HardCase]
    version: str
    last_updated: str

class DataFlywheel:
    def __init__(self):
        self.eval_sets = {}  # name -> EvalSet
        self.hard_cases = []  # list[HardCase]
        self.active_cases = []  # list[HardCase]

    def add_hard_case(self, case: HardCase):
        """加入 hard case"""
        self.hard_cases.append(case)

    def sample_hard_cases(self, count: int = 20) -> list[HardCase]:
        """采样 hard cases（faithfulness < 0.5）"""
        return [c for c in self.hard_cases if c.faithfulness < 0.5][:count]

    def create_eval_set(self, name: str, cases: list[HardCase], version: str = 'v1.0') -> EvalSet:
        """创建 eval set"""
        eval_set = EvalSet(name=name, cases=cases, version=version, last_updated=time.strftime('%Y-%m-%d'))
        self.eval_sets[name] = eval_set
        return eval_set

    def update_eval_set(self, name: str, new_cases: list[HardCase], version: str = None):
        """更新 eval set（加入新 cases）"""
        if name in self.eval_sets:
            eval_set = self.eval_sets[name]
            eval_set.cases.extend(new_cases)
            eval_set.version = version or eval_set.version + '.1'
            eval_set.last_updated = time.strftime('%Y-%m-%d')
        else:
            self.create_eval_set(name, new_cases, version or 'v1.0')

# 初始化数据飞轮
flywheel = DataFlywheel()
print('数据飞轮已初始化（DataFlywheel 类）')

## 2. CI 集成 —— 自动化评估

**CI 集成的目标**：每次 PR 自动跑 eval，防止退化

**CI 集成的 3 个组件**：

1. **Pre-commit Hooks（提交前检查）**
- 提交前自动跑 eval，检查指标是否退化
- 用 `pre-commit` 管理 hooks
- 示例：`pre-commit run eval --all-files`

2. **GitHub Actions（PR 检查）**
- 每次 PR 自动跑 eval（30 题）
- 生成 baseline.md，对比上一个 PR 的 metrics
- 如果 delta < 0（指标退化），阻止合并

3. **Auto-merge（自动合并）**
- 如果 delta >= 0（指标改善或持平），自动合并
- 如果 delta < 0（指标退化），标记为 failure

**CI 集成的示例 workflow**：
```yaml
name: RAG Eval CI
on:
  pull_request:
    types: [opened, synchronize]
jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run eval
        run: python eval_ragas.py --eval_set eval/v1_30q.jsonl --output metrics.json
      - name: Compare with baseline
        run: python compare_metrics.py --current metrics.json --baseline metrics/baseline.json
      - name: Upload metrics
        uses: actions/upload-artifact@v3
        with:
          name: metrics
          path: metrics.json
```

In [ ]:
# CI 集成示例

class CIEvaluator:
    def __init__(self, baseline_file: str):
        self.baseline_file = baseline_file
        self.current_metrics = {}
        self.baseline_metrics = {}

    def load_baseline(self):
        """加载 baseline metrics"""
        with open(self.baseline_file, 'r') as f:
            self.baseline_metrics = json.load(f)

    def run_eval(self, eval_set: list[dict], system: RAGSystem) -> dict:
        """运行 eval（30 题）"""
        results = evaluate_ragas(system, eval_set)
        return results['agg']

    def compare(self, current: dict, baseline: dict) -> dict:
        """对比 current 和 baseline，返回 delta"""
        delta = {}
        for k in baseline:
            delta[k] = current.get(k, 0.0) - baseline[k]
        return delta

    def check_pass(self, delta: dict) -> bool:
        """检查 delta 是否通过（delta >= 0 表示改善或持平）"""
        for k, v in delta.items():
            if v < 0:  # 指标退化
                return False
        return True

    def generate_report(self, delta: dict) -> str:
        """生成 CI report"""
        report = """=== RAG Eval CI Report ===

Checks:
  Context Precision: {' +0.02' if delta.get('context_precision', 0) >= 0 else ' -0.03'}
  Faithfulness: {' +0.01' if delta.get('faithfulness', 0) >= 0 else ' -0.02'}
  Answer Relevance: {' +0.03' if delta.get('answer_relevance', 0) >= 0 else ' -0.01'}

Result:
  {'PASS' if self.check_pass(delta) else 'FAIL'}
"""
        return report

# 初始化 CI evaluator
ci_eval = CIEvaluator(baseline_file='metrics/baseline.json')
print('CI 集成评估器已初始化（CIEvaluator 类）')

## 3. Judge 校准 —— 提升 LLM-as-judge 的稳定性

**Judge 校准的目标**：提升 LLM-as-judge 的准确性和稳定性

**Judge 校准的 2 种方法**：

1. **Self-Consistency（自洽性）**
- 多次采样（如 3-5 次），取平均
- 用相同的 rubric，但不同的 prompt（如不同的 template）
- 取 mean score，提高稳定性

2. **Confidence Calibration（置信度校准）**
- 训练一个校准模型（如 logistic regression），将 LLM 的 raw score 映射到 calibrated score
- LLM 输出 raw score（0-1），校准模型输出 calibrated score
- Calibrated score 更接近真实标签

**Judge 校准的步骤**：
1. **收集数据**：收集 100 条 LLM-as-judge 的 raw score + ground truth
2. **训练校准模型**：用 logistic regression 校准 raw score
3. **验证**：对比 calibrated score 和 raw score 的 accuracy
4. **部署**：在生产中用 calibrated score

**Self-Consistency 的优势**：
- 不需要标注数据（无监督）
- 简单易实现
- 提高稳定性（减少随机性）

**Confidence Calibration 的优势**：
- 提高准确性（更接近真实标签）
- 可以检测 outlier（如 LLM 输出极端的 score）
- 可以过滤低 confidence 的 samples（用于 active learning）

In [ ]:
# Judge 校准示例（Self-Consistency）

def llm_judge_self_consistency(question: str, answer: str, contexts: list[str], 
                               rubric: str, n_samples: int = 3) -> float:
    """LLM-as-judge + self-consistency（多采样取平均）"""
    scores = []
    for _ in range(n_samples):
        # 实际调用 LLM（这里用 stub 模拟）
        if MODE == 'ONLINE':
            resp = chat(f'{rubric}\n问题: {question}\n回答: {answer}\n评分: (只输出数字)')
        else:
            # OFFLINE：用规则模拟
            resp = f'{np.random.uniform(0.5, 1.0):.2f}'
        m = re.search(r'0\.?\d+', resp)
        scores.append(float(m.group()) if m else 0.5)
    return float(np.mean(scores))

# Judge 校准示例（Confidence Calibration）

class ConfidenceCalibrator:
    def __init__(self):
        self.calibrator = None  # logistic regression model

    def train(self, raw_scores: list[float], labels: list[int]) -> None:
        """训练校准模型（logistic regression）"""
        # 实际会用 sklearn 的 LogisticRegression
        # self.calibrator = LogisticRegression().fit(raw_scores.reshape(-1, 1), labels)
        pass

    def calibrate(self, raw_score: float) -> float:
        """校准 raw score"""
        # 实际调用 calibrator.predict(raw_score.reshape(1, -1))[0]
        if self.calibrator:
            # return self.calibrator.predict(raw_score.reshape(1, -1))[0]
            pass
        else:
            return raw_score  # 未训练，直接返回 raw score

    def evaluate(self, raw_scores: list[float], labels: list[int]) -> float:
        """评估校准效果（accuracy）"""
        # 实际会用 sklearn 的 accuracy_score
        calibrated = [self.calibrate(r) for r in raw_scores]
        # return accuracy_score(labels, calibrated)
        return 0.0

# 初始化校准器
calibrator = ConfidenceCalibrator()
print('Judge 校准器已初始化（ConfidenceCalibrator 类）')

## 4. 完整数据飞轮流程 —— 从 monitor 到 deploy

**完整流程**：
1. **Monitor（监控 live traffic）**：收集 hard cases
2. **Sample（采样）**：抽取 20 条 hard cases
3. **Label（标注）**：人工标注 label
4. **Add（加入 eval set）**：更新 eval set
5. **Evaluate（迭代验证）**：跑 eval，看指标是否改善
6. **CI（自动化检查）**：每次 PR 自动跑 eval
7. **Judge Calibration（校准 judge）**：提升 LLM-as-judge 的稳定性
8. **Deploy（部署）**：部署 winner

**Monitor 阶段**：
- 从 live traffic 中采样 low faithfulness 的 case（< 0.5）
- 记录到 hard_cases 列表

**Sample + Label 阶段**：
- 抽取 20 条 hard cases
- 人工标注 label（正确答案 / 为什么错）

**Add + Evaluate 阶段**：
- 加入 eval set
- 跑 eval，看指标是否改善

**CI + Judge Calibration 阶段**：
- 每次 PR 自动跑 eval（用 CI evaluator）
- 对比 delta，如果 delta < 0，阻止合并
- 校准 LLM-as-judge（self-consistency + confidence calibration）

In [ ]:
# 完整数据飞轮示例

# 1. Monitor 阶段：从 baseline_eval 中采样 hard cases
hard_cases = flywheel.sample_hard_cases(count=20)

print(f'从 baseline_eval 采样 {len(hard_cases)} 条 hard cases（faithfulness < 0.5）')

# 2. Label 阶段：人工标注（这里用 mock）
for case in hard_cases:
    case.label = '人工标注：答案有幻觉，缺少关键信息'
    case.timestamp = time.time()

print(f'✅ 人工标注完成（{len(hard_cases)} 条）')

# 3. Add 阶段：加入 eval set
flywheel.update_eval_set(name='eval_v2', new_cases=hard_cases, version='v2.0')

print(f'✅ eval set 已更新（version: v2.0, {len(flywheel.eval_sets["eval_v2"].cases)} cases）')

# 4. Evaluate 阶段：用更新后的 eval set 跑 eval
new_eval_set = flywheel.eval_sets['eval_v2'].cases

def case_to_dict(case: HardCase) -> dict:
    return {
        'q': case.question,
        'expect': [],  # 人工标注的 label
        'gold_answer': case.label
}

new_eval_dict = [case_to_dict(c) for c in new_eval_set]

new_eval = evaluate_ragas(baseline, new_eval_dict)

print('\n===== New Eval (v2.0) =====')
for k, v in new_eval['agg'].items():
    print(f'  {k:<25} = {v:.3f}')

# 5. CI 阶段：对比 baseline 和 new eval
ci_eval.load_baseline()
current_metrics = baseline_eval['agg']

delta = ci_eval.compare(current_metrics, ci_eval.baseline_metrics)

print('\n===== CI Delta (new - baseline) =====')
for k, v in delta.items():
    print(f'  {k:<25} = {v:+.3f}')

print(f'\n===== CI Result =====')
if ci_eval.check_pass(delta):
    print('  ✅ PASS - 指标改善或持平，可以合并')
else:
    print('  ❌ FAIL - 指标退化，阻止合并')

# 6. Judge Calibration 阶段：校准 LLM-as-judge

print('\n===== Judge Calibration =====')
print('  Self-Consistency: 3 samples, take mean')
print('  Confidence Calibration: logistic regression (待训练)')

## 5. 实战建议 —— 数据飞轮的最佳实践

**数据飞轮的 3 个原则**：
1. **主动收集**：从 live traffic 中收集 hard cases（低 faithfulness / 低 answer_correctness）
2. **人工标注**：人工标注 hard cases 的 label（正确答案 / 为什么错）
3. **持续迭代**：定期更新 eval set，提高覆盖度

**CI 集成的最佳实践**：
- Pre-commit hooks：提交前自动跑 eval
- GitHub Actions：PR 自动跑 eval
- Auto-merge：delta >= 0 才自动合并

**Judge 校准的最佳实践**：
- Self-Consistency：多采样（3-5 次），取 mean
- Confidence Calibration：用 logistic regression 校准 raw score
- 验证：对比 calibrated score 和 raw score 的 accuracy

**推荐 pipeline**：
```
1. Monitor（监控 live traffic）
   → 收集 hard cases（faithfulness < 0.5）
2. Sample（采样）
   → 抽取 20-50 条 hard cases
3. Label（标注）
   → 人工标注 label
4. Add（加入 eval set）
   → 更新 eval set，提高覆盖度
5. Evaluate（迭代验证）
   → 跑 eval，看指标是否改善
6. CI（自动化检查）
   → 每次 PR 自动跑 eval
7. Judge Calibration（校准 judge）
   → self-consistency + confidence calibration
8. Deploy（部署）
   → 部署 winner
```

In [ ]:
# 生成数据飞轮报告

report = f"""
=== Data Flywheel Report ===

Monitor:
  Hard cases collected: {len(hard_cases)}
  Low faithfulness (< 0.5): {len(hard_cases)}

Add:
  Eval set updated: eval_v2
  Version: v2.0
  Cases: {len(flywheel.eval_sets["eval_v2"].cases)}

Evaluate:
  Context Precision: {new_eval["agg"]["context_precision"]:.3f}
  Faithfulness: {new_eval["agg"]["faithfulness"]:.3f}
  Answer Relevance: {new_eval["agg"]["answer_relevance"]:.3f}

CI:
  Delta: {ci_eval.check_pass(delta)}
  Status: {'PASS' if ci_eval.check_pass(delta) else 'FAIL'}

Judge Calibration:
  Self-Consistency: 3 samples, take mean
  Confidence Calibration: logistic regression (待训练)

Next Steps:
  1. Train confidence calibrator on 100 samples
  2. Validate calibrated score accuracy
  3. Deploy updated eval set
"""

print(report)

print('\n→ 实战建议：每次改动都写一份数据飞轮报告，主动收集 hard cases，持续迭代 eval set。')

## 深入思考

1. **数据飞轮的 3 种类型是什么？**
   - Active Learning（主动学习）：从 live traffic 中采样 low faithfulness 的 case，人工标注，加入 eval set。
   - Adversarial Examples（对抗样本）：生成对抗 query（同义改写 / 反问 / 歧义问题），人工标注，加入 eval set，提高鲁棒性。
   - Hard Cases（困难样本）：从 eval set 中找出 high faithfulness 但 low context_precision 的 case，分析为什么 recall 烂，优化 retrieval 策略。
2. **CI 集成的 3 个组件是什么？**
   - Pre-commit Hooks：提交前自动跑 eval，防止退化。
   - GitHub Actions：PR 自动跑 eval，生成 baseline.md。
   - Auto-merge：delta >= 0 才自动合并，delta < 0 阻止合并。
3. **Judge 校准的 2 种方法是什么？**
   - Self-Consistency（自洽性）：多采样（3-5 次），取 mean，提高稳定性。
   - Confidence Calibration（置信度校准）：训练校准模型（logistic regression），将 LLM 的 raw score 映射到 calibrated score，提高准确性。
4. **为什么需要数据飞轮？**
   - RAG 评估需要持续的反馈循环：每次改动 → eval 找到 weak cases → 人工标注 → 更新 eval set → 下次 eval 更准 → 验证改动更有效 → 飞轮加速迭代。
5. **为什么需要 Judge 校准？**
   - LLM-as-judge 有 bias 和随机性（自洽性不够），通过 self-consistency 和 confidence calibration 提高准确性和稳定性，减少误判。

**改一改**：
- 在 ONLINE 模式跑 self-consistency，对比单次采样和 self-consistency 的稳定性
- 添加 confidence calibration：训练一个 logistic regression 模型，校准 LLM 的 raw score

## 自检 ✅

- [ ] 解释数据飞轮的 3 种类型（active learning / adversarial examples / hard cases）。
- [ ] 解释 CI 集成的 3 个组件（pre-commit hooks / GitHub Actions / auto-merge）。
- [ ] 解释 Judge 校准的 2 种方法（self-consistency / confidence calibration）。
- [ ] 解释「为什么需要数据飞轮」。
- [ ] 解释「为什么需要 Judge 校准」。

## 🎉 46 完成

**走完 16-24 + 44 + 45 + 46 你应该具备**：
- ✅ 手撸数据飞轮（active learning / adversarial examples / hard cases）
- ✅ 接入 CI 集成（pre-commit hooks / GitHub Actions / auto-merge）
- ✅ 实现 Judge 校准（self-consistency / confidence calibration）
- ✅ 完成完整的数据飞轮流程（monitor → sample → label → add → evaluate → ci → judge calibration → deploy）
- ✅ 生成数据飞轮报告，指导迭代

**恭喜你完成 01-RAG 全部 24 + 3 个高级 notebook！**

**下一步**：→ 进入 [02-Agent](../../../02-Agent/) 或 [04-模型微调](../../../04-模型微调-Finetuning/)